# RadGraph F1 mectics

**RadGraph** (Radiology Graph) is a specialized NLP metric that evaluates RRG by converting text into a clinical **knowledge graph** and measuring the harmonic mean of accurately matched medical entities and their anatomical relationships.

To install RadGraph F1 metrics:

    # Create virtual environment
    conda create --name metrics python=3.13.*
    conda activate metrics

    # For old CUDA Version: 12.4 (our DL4 server)
    pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

    # Install RadGraph with old transformers version 4.57.6
    pip install radgraph "transformers<5.0.0"

In [1]:
GPU = 3  # GPU number. After change, restart the kernel

%set_env CUDA_VISIBLE_DEVICES={GPU}

import os
import torch

if torch.cuda.is_available():  # make sure GPU is available
    num = torch.cuda.device_count()
    print(f"GPU count: {num}")
    for i in range(num):
        print(f"GPU {i} name: {torch.cuda.get_device_name(i)}")
else:
    print("GPU is not available")


# Disable Hugging Face tokenizer forks warning when using multiprocessing
os.environ["TOKENIZERS_PARALLELISM"] = "false"

env: CUDA_VISIBLE_DEVICES=3
GPU count: 1
GPU 0 name: Tesla V100-PCIE-16GB


In [2]:
%%writefile radgraph_workers.py

# Run this cell in your Jupyter Notebook.
# The %%writefile magic command will automatically create the
# `radgraph_workers.py` file in the same folder as your notebook:

import os
import sys
from radgraph import F1RadGraph


# Global initializer for workers.
# Each background worker will spawn its own instance exactly once.
_GLOBAL_METRIC = None


def init_worker():
    """ Initializes the F1RadGraph model once inside each spawned background worker. """
    global _GLOBAL_METRIC
    # Suppress the "Using device: cuda:0" internal print statements inside the workers
    old_stdout = sys.stdout
    sys.stdout = open(os.devnull, "w")
    try:
        # Initialize the metric using the modern-radgraph-xl checkpoint from Stanford
        # reward_level="all" returns entity_f1, relation_f1, and partial_relation_f1
        _GLOBAL_METRIC = F1RadGraph(reward_level="all", model_type="modern-radgraph-xl")
    finally:
        sys.stdout.close()
        sys.stdout = old_stdout


def process_batch(batch_data):
    """ The core execution block handled by each background worker process. """
    b_hyps, b_refs = batch_data
    # Use the persistent model instance initialized within this process's memory space
    mean_reward, _, _, _ = _GLOBAL_METRIC(hyps=b_hyps, refs=b_refs)
    return mean_reward

Overwriting radgraph_workers.py


In [3]:
import os
import sys
import math
import torch
import numpy as np
import pandas as pd
import multiprocessing as mp

from tqdm.notebook import tqdm
from radgraph import F1RadGraph  # official Stanford F1RadGraph evaluator

# Import worker functions from the file on disk.
# This completely bypasses the Jupyter AttributeError
from radgraph_workers import init_worker, process_batch


BATCH_SIZE = 400  # 4660/400 = 12 batches for 4 workers: 12/4 = 3 cycles

CPU = mp.cpu_count()  # total number of CPU cores
# NUM_WORKERS = math.ceil(CPU * 1/6)  # number of CPU cores to calculate (32/6 = 6)
NUM_WORKERS = 4  # hardcore parallel processes to 4
print(f"Total number of CPU cores: {CPU}")
print(f"Number of worker cores: {NUM_WORKERS}")


def calculate_radgraph_f1(
    predicted_list,
    actual_list,
    batch_size = BATCH_SIZE,
    show = True,
):
    """
    Computes the official RadGraph F1 metrics in parallel using a Multiprocessing Pool.
    Fully compatible with modern Torch, CUDA, and text parsing configurations.
    """
    # 1. Chunk the dataset into pairs of (hypotheses, references)
    batches = []
    for i in range(0, len(predicted_list), batch_size):
        b_hyps = predicted_list[i : i + batch_size]
        b_refs = actual_list[i : i + batch_size]
        batches.append((b_hyps, b_refs))

    all_rewards = []

    # 2. Spin up a multiprocessing pool using our specialized background initializer
    ctx = mp.get_context("spawn")  # use the "spawn" context to handle CUDA across processes safely
    with ctx.Pool(processes=NUM_WORKERS, initializer=init_worker) as pool:
        # pool.imap allows us to easily wrap the parallel execution in a tqdm progress bar
        iterator = pool.imap(process_batch, batches)

        for mean_reward in tqdm(
            iterator,
            total = len(batches),
            desc = "Evaluating Reports (Parallel)",
            leave = show,
        ):
            all_rewards.append(mean_reward)

    # 3. Calculate final global mean across all processed batches safely
    final_mean = np.mean(all_rewards, axis=0)
    rg_e, rg_er, rg_bar_er = final_mean

    if show:
        print("\n--- RadGraph F1 Evaluation Metrics ---")
        print(f"Entity F1 (Clinical Entities Only):       {rg_e:.4f}")
        print(f"Relation F1 (Entities + Exact Relations): {rg_er:.4f}  <-- Standard RadGraph F1")
        print(f"Partial Relation F1 (Normalized Match):   {rg_bar_er:.4f}")

    return rg_er


def test():
    """ Test function """
    # Ground-truth annotations (References)
    actual_reports = [
        "no acute cardiopulmonary abnormality",
        "endotracheal tube is present and bibasilar opacities likely represent mild atelectasis"
    ]
    # Model predictions (Hypotheses)
    predicted_reports = [
        "no acute cardiopulmonary abnormality",
        "et tube terminates 2 cm above the carina and bibasilar opacities"
    ]
    # Execute the official clean pipeline
    calculate_radgraph_f1(predicted_reports, actual_reports)


def calculate(csv_file):
    """ Calculate RadGraph F1 mectics """
    df = pd.read_csv(csv_file)
    # display(df.head(3))  # show first rows

    actual_reports = df["actual_text"].tolist()
    columns_list = df.columns.tolist()

    # Outer loop progress tracking for your models
    for col in tqdm(columns_list[2:], desc="Overall Model Progress"):
        predicted_reports = df[col].tolist()
        radgraph = calculate_radgraph_f1(predicted_reports, actual_reports, show=False)
        print(f"\t{col}: {radgraph:.3f}")


if __name__ == "__main__":  # CRITICAL FOR NOTEBOOKS: Ensure the main entry point is guarded
    # test()  # preliminary test
    calculate("../test_all.csv")  # calculate all

Total number of CPU cores: 32
Number of worker cores: 4


Overall Model Progress:   0%|          | 0/17 [00:00<?, ?it/s]

Evaluating Reports (Parallel):   0%|          | 0/10 [00:00<?, ?it/s]

	test_base_qwen: 0.021


Evaluating Reports (Parallel):   0%|          | 0/10 [00:00<?, ?it/s]

...

	best_model_lstm_5_layers_resnet_unfreeze_top_p=0.1: 0.224


Evaluating Reports (Parallel):   0%|          | 0/10 [00:00<?, ?it/s]

	best_model_lstm_5_layers_resnet_unfreeze_top_p=0.4: 0.220


Evaluating Reports (Parallel):   0%|          | 0/10 [00:00<?, ?it/s]

	best_model_lstm_5_layers_resnet_unfreeze_top_p=0.5: 0.216
